# Phase 3 — fine-tune

**33.2 sec/kimg** measured on 2x T4 with `batch_gpu=32`, so 300 kimg is **2.8 h**
per class and up to four classes fit one 12 h session.

## Run as a saved version

> **Save Version → Save & Run All (Commit)**

A browser tab will not survive this. **GPU T4 x2**, **Internet On**,
`stylegan.zip` attached.

| class | images | share of the 400-image target |
|---|---|---|
| `mammalian` | 789 | 37% |
| `arthropod` | 237 | 11% |
| `plant_fungus` | 197 | 9% |

All three in one session is 8.3 h. Judge the grids before spending the other
seven classes.

## 1. What to set

In [ ]:
CLASSES   = ["mammalian"]   # add "arthropod", "plant_fungus" to chain in one run
KIMG      = 300             # 2.8 h per class
GPUS      = 2               # 2.14x faster than 1; needs the patches in step 2
BATCH_GPU = 32              # 64 // (32 * 2) = 1 accumulation round
FREEZED   = 0               # first knob to try if results disappoint
RESUME    = None            # or a snapshot path to continue a class

SEC_PER_KIMG = 33.2
hours = len(CLASSES) * KIMG * SEC_PER_KIMG / 3600
print(f"{len(CLASSES)} class(es) x {KIMG} kimg = {hours:.1f} h")
assert hours < 11, "over the 12 h session cap - drop a class or lower KIMG"

## 2. Setup

Seven patches for torch 2.x; `patch()` asserts each target exists, so a silent
no-op is impossible. Reasoning in `docs/stylegan.md`.

The last two matter only for `GPUS=2`: `batch_gpu` is pinned to NVlabs' 8-GPU
reference rig, and the ranks otherwise disagree on `noise_const` because
`--resume` loads on rank 0 alone.

In [ ]:
import os, sys, json, time, pathlib, subprocess, shutil
import torch

REPO = "/kaggle/working/stylegan2-ada-pytorch"
shutil.rmtree(REPO, ignore_errors=True)
subprocess.run(["git", "clone", "-q",
                "https://github.com/NVlabs/stylegan2-ada-pytorch.git", REPO], check=True)
sys.path.insert(0, REPO)
NL = chr(10)

def patch(rel, old, new):
    f = pathlib.Path(REPO) / rel
    s = f.read_text()
    assert old in s, f"patch target not found in {rel}"
    f.write_text(s.replace(old, new))

# Kernels report "Failed!" after building fine.
patch("torch_utils/custom_ops.py",
      "torch.utils.cpp_extension.load(name=module_name",
      "module = torch.utils.cpp_extension.load(name=module_name")
patch("torch_utils/custom_ops.py",
      "        module = importlib.import_module(module_name)" + NL, "")

# TypeError: object.__init__() takes exactly one argument
patch("torch_utils/misc.py", "super().__init__(dataset)", "super().__init__()")

# R1 needs grid_sample's second derivative, which torch still lacks.
GSG = "torch_utils/ops/grid_sample_gradfix.py"
patch(GSG, "any(torch.__version__.startswith(x) for x in ['1.7.', '1.8.', '1.9'])",
      "True")
patch(GSG,
      "op = torch._C._jit_get_operation('aten::grid_sampler_2d_backward')" + NL +
      "        grad_input, grad_grid = op(grad_output, input, grid, 0, 0, False)",
      "op = torch.ops.aten.grid_sampler_2d_backward" + NL +
      "        grad_input, grad_grid = op(grad_output, input, grid, 0, 0, False, [True, True])")

# batch_gpu is pinned to mb // 8 (NVlabs' rig, not ours). Default unchanged.
patch("train.py", "args.batch_gpu = spec.mb // spec.ref_gpus",
      "args.batch_gpu = int(os.environ.get('BATCH_GPU', spec.mb // spec.ref_gpus))")

# Multi-GPU only: ranks disagree on noise_const. Sync once from rank 0.
patch("training/training_loop.py",
      "    # Print network summary tables.",
      NL.join(["    if num_gpus > 1:",
               "        torch.cuda.set_device(device)",
               "        for _m in [G, D, G_ema]:",
               "            for _, _t in misc.named_params_and_buffers(_m):",
               "                torch.distributed.broadcast(_t, src=0)",
               "",
               "    # Print network summary tables."]))

DATA = next(p.parent for p in pathlib.Path("/kaggle/input").glob("**/summary.json"))
URL = ("https://nvlabs-fi-cdn.nvidia.com/stylegan2-ada-pytorch/pretrained/"
       "transfer-learning-source-nets/lsundog-res256-paper256-kimg100000-noaug.pkl")
PKL = "/kaggle/working/lsundog-res256.pkl"
if not os.path.exists(PKL):
    subprocess.run(["wget", "-q", "-O", PKL, URL], check=True)

print("torch", torch.__version__, "|", torch.cuda.get_device_name(0),
      "|", torch.cuda.device_count(), "gpus | 7 patches applied")

## 3. Train

`--cfg=paper256` must match the source net or `--resume` fails on layer shapes.
`--snap=10` gives ~8 checkpoints per class; **the best is rarely the last**.

`--metrics=none` on purpose — KID belongs in Phase 4, and `kid50k_full`
generates 50k samples per evaluation.

In [ ]:
NOISE = ("conv2d_gradfix", "Grad strides", "grad.sizes()", "bucket_view.sizes()")
runs = {}

for cls in CLASSES:
    print(f"{'='*60}{NL}{cls}{NL}{'='*60}")
    zp = f"/kaggle/working/{cls}.zip"
    subprocess.run([sys.executable, f"{REPO}/dataset_tool.py",
                    f"--source={DATA / cls}", f"--dest={zp}"], check=True)

    outdir = f"/kaggle/working/{cls}_run"
    cmd = [sys.executable, f"{REPO}/train.py",
           f"--outdir={outdir}", f"--data={zp}", f"--gpus={GPUS}",
           "--cfg=paper256", "--mirror=1", "--aug=ada", "--target=0.6",
           f"--resume={RESUME or PKL}", "--snap=10", "--metrics=none",
           f"--kimg={KIMG}"]
    if FREEZED:
        cmd.append(f"--freezed={FREEZED}")

    t0 = time.time()
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1,
                         env=dict(os.environ, BATCH_GPU=str(BATCH_GPU)))
    for line in p.stdout:
        if not any(n in line for n in NOISE):
            print(line, end="")
    p.wait()

    if p.returncode != 0:
        print(f"!!! {cls} FAILED (exit {p.returncode}) - continuing")
        continue
    runs[cls] = outdir
    print(f"{cls} done in {(time.time()-t0)/3600:.2f} h")

print(NL, "trained:", list(runs))

## 4. Results

The grids are the honest signal: the deliverable is a game judged by eye, so KID
in Phase 4 only ranks snapshots that already look plausible.

`fakes_init.png` is the dog starting point. Later grids should look like the class.

In [ ]:
import matplotlib.pyplot as plt
import PIL.Image

for cls, outdir in runs.items():
    run = sorted(pathlib.Path(outdir).glob("00000-*"))[-1]
    grids = sorted(run.glob("fakes*.png"))
    snaps = sorted(run.glob("network-snapshot-*.pkl"))

    for g in [grids[0], grids[len(grids) // 2], grids[-1]]:
        im = PIL.Image.open(g)
        im.thumbnail((1100, 1100))
        plt.figure(figsize=(13, 13 * im.height / im.width))
        plt.imshow(im)
        plt.axis("off")
        plt.title(f"{cls} - {g.name}")
        plt.show()

    ticks = [json.loads(l) for l in (run / "stats.jsonl").read_text().splitlines() if l.strip()]
    get = lambda t, k: t.get(k, {}).get("mean", 0)
    print(f"{cls}: {len(snaps)} snapshots, {sum(s.stat().st_size for s in snaps)/1e9:.1f} GB")
    for t in ticks[::max(1, len(ticks) // 8)]:
        print(f"  kimg {get(t,'Progress/kimg'):6.0f}  G {get(t,'Loss/G/loss'):7.3f}"
              f"  D {get(t,'Loss/D/loss'):7.3f}  ada_p {get(t,'Progress/augment'):.3f}")
    print()

## Done — then what

1. **Look at the last grid.** Do they read as creatures of this class?
2. If `ada_p` passed ~0.7, the discriminator was straining — that is the
   argument for trying FreezeD next.
3. At 2.8 h per class the remaining seven cost ~19 h, so all ten fit one
   ~30 GPU-h week.

Snapshots are in this notebook's **Output**, ~350 MB each. Keep the best per class.